# 01 — Dataset Download

Downloads **GRAPE** and **UWHVF** — the two open-access datasets best suited for glaucoma VF progression research.

| Dataset | Records | Modalities | Labels |
|---------|---------|------------|--------|
| GRAPE | 1,115 follow-up / 263 eyes | VF + Fundus + OCT + Clinical | **VF progression annotations** |
| UWHVF | 28,943 VFs / 3,871 patients | VF only | Raw sensitivity + MD + VFI |

After download, raw data lands in `data/raw/`. Preprocessing is in `04_vf_preprocessing.ipynb`.

In [10]:
import subprocess
import sys
import os
import zipfile
import json
from pathlib import Path
import urllib.request

ROOT = Path("..")
RAW  = ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)
print("Data dir:", RAW.resolve())

Data dir: /Users/taiaburrahman/Desktop/git/glam/data/raw


## 1. UWHVF — University of Washington Humphrey VF Dataset

28,943 visual fields from 3,871 patients. VF-only (no images), but large and longitudinal.  
Great for pretraining the temporal model on VF progression patterns.

**Source**: https://github.com/uw-biomedical-ml/uwhvf

In [11]:
UWHVF_DIR = RAW / "uwhvf"

if not UWHVF_DIR.exists():
    print("Cloning UWHVF...")
    result = subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/uw-biomedical-ml/uwhvf.git",
         str(UWHVF_DIR)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    else:
        print("Done.")
else:
    print("UWHVF already present — pulling latest")
    subprocess.run(["git", "-C", str(UWHVF_DIR), "pull"], capture_output=True)

UWHVF already present — pulling latest


In [12]:
# Inspect UWHVF structure
print("\nUWHVF files:")
for p in sorted(UWHVF_DIR.rglob("*"))[:30]:
    if p.is_file():
        print(f"  {p.relative_to(UWHVF_DIR)}  ({p.stat().st_size/1024:.1f} KB)")


UWHVF files:
  .git/FETCH_HEAD  (0.1 KB)
  .git/HEAD  (0.0 KB)
  .git/ORIG_HEAD  (0.0 KB)
  .git/config  (0.3 KB)
  .git/description  (0.1 KB)
  .git/hooks/applypatch-msg.sample  (0.5 KB)
  .git/hooks/commit-msg.sample  (0.9 KB)
  .git/hooks/fsmonitor-watchman.sample  (4.6 KB)
  .git/hooks/post-update.sample  (0.2 KB)
  .git/hooks/pre-applypatch.sample  (0.4 KB)
  .git/hooks/pre-commit.sample  (1.6 KB)
  .git/hooks/pre-merge-commit.sample  (0.4 KB)
  .git/hooks/pre-push.sample  (1.3 KB)
  .git/hooks/pre-rebase.sample  (4.8 KB)
  .git/hooks/pre-receive.sample  (0.5 KB)
  .git/hooks/prepare-commit-msg.sample  (1.5 KB)
  .git/hooks/push-to-checkout.sample  (2.7 KB)
  .git/hooks/sendemail-validate.sample  (2.3 KB)
  .git/hooks/update.sample  (3.6 KB)
  .git/index  (1.9 KB)
  .git/info/exclude  (0.2 KB)
  .git/logs/HEAD  (0.2 KB)
  .git/logs/refs/heads/master  (0.2 KB)


In [13]:
# Quick peek at UWHVF data format
import json

# UWHVF stores data as JSON
json_files = list(UWHVF_DIR.rglob("*.json"))
print(f"Found {len(json_files)} JSON files")

if json_files:
    with open(json_files[0]) as f:
        sample = json.load(f)
    print("\nSample keys:", list(sample.keys()) if isinstance(sample, dict) else type(sample))
    if isinstance(sample, list):
        print("First record:", json.dumps(sample[0], indent=2)[:500])

Found 2 JSON files

Sample keys: ['$schema', '$id', 'title', 'description', 'type', 'properties']


## 2. GRAPE — Multi-modal Glaucoma Progression Dataset

Best dataset for our task: has VF + fundus + OCT + clinical + **progression annotations**.

**Data**: Hosted on **Figshare** (the GitHub repo `li-research/GRAPE` is no longer available).  
**Paper**: *GRAPE: A multi-modal dataset of longitudinal follow-up visual field and fundus images for glaucoma management*, Nature Scientific Data, 2023 — [article](https://www.nature.com/articles/s41597-023-02424-4) (see "Data availability" for Figshare link).  

Download the dataset from Figshare and extract it into `data/raw/grape/`.

In [14]:
GRAPE_DIR = RAW / "grape"

if not GRAPE_DIR.exists():
    print("GRAPE directory not found.")
    print()
    print("GRAPE is hosted on Figshare — the GitHub repo (li-research/GRAPE) no longer exists.")
    print("Manual download steps:")
    print("  1. Visit the Figshare link in the paper's Data Availability section:")
    print("     https://www.nature.com/articles/s41597-023-02424-4")
    print("  2. Download the dataset files")
    print(f"  3. Extract into: {GRAPE_DIR.resolve()}")
    print()
    print("Alternatively, use the Figshare API if you have the article DOI/ID.")
else:
    print("GRAPE directory found:")
    for p in sorted(GRAPE_DIR.iterdir())[:10]:
        print(f"  {p.name}")

GRAPE directory not found.

GRAPE is hosted on Figshare — the GitHub repo (li-research/GRAPE) no longer exists.
Manual download steps:
  1. Visit the Figshare link in the paper's Data Availability section:
     https://www.nature.com/articles/s41597-023-02424-4
  2. Download the dataset files
  3. Extract into: /Users/taiaburrahman/Desktop/git/glam/data/raw/grape

Alternatively, use the Figshare API if you have the article DOI/ID.


In [15]:
# Check GRAPE structure and README for download instructions
print("GRAPE files:")
if GRAPE_DIR.exists():
    for p in sorted(GRAPE_DIR.iterdir()):
        print(f"  {p.name}")
    readme = GRAPE_DIR / "README_GRAPE_DOWNLOAD.txt"
    if not readme.exists():
        readme = GRAPE_DIR / "README.md"
    if readme.exists():
        print("\n--- README (first 2000 chars) ---")
        print(readme.read_text()[:2000])
else:
    print("  (directory not found)")

GRAPE files:
  (directory not found)


In [16]:
# Run GRAPE download script if available
download_scripts = list(GRAPE_DIR.rglob("download*.py")) + list(GRAPE_DIR.rglob("download*.sh"))
print("Download scripts found:", download_scripts)

# If no script, check for direct data files already present
csv_files = list(GRAPE_DIR.rglob("*.csv"))
print(f"CSV files: {len(csv_files)}")
for f in csv_files[:5]:
    print(f"  {f.relative_to(GRAPE_DIR)}")

Download scripts found: []
CSV files: 0


## 3. Verify Downloads

In [17]:
def dataset_summary(path: Path, name: str):
    if not path.exists():
        print(f"[{name}] NOT FOUND at {path}")
        return
    files = list(path.rglob("*"))
    data_files = [f for f in files if f.is_file()]
    total_mb = sum(f.stat().st_size for f in data_files) / (1024**2)
    print(f"[{name}] {len(data_files)} files, {total_mb:.1f} MB")
    exts = {}
    for f in data_files:
        exts[f.suffix] = exts.get(f.suffix, 0) + 1
    print(f"  Extensions: {dict(sorted(exts.items(), key=lambda x: -x[1])[:5])}")

dataset_summary(UWHVF_DIR, "UWHVF")
dataset_summary(GRAPE_DIR, "GRAPE")

[UWHVF] 53 files, 137.9 MB
  Extensions: {'': 15, '.csv': 15, '.sample': 14, '.md': 3, '.json': 2}
[GRAPE] NOT FOUND at ../data/raw/grape


## 4. Harvard Glaucoma Dataset (Optional)

Multimodal + multitask. Check availability at ICCV 2023 paper release page.

In [18]:
HARVARD_DIR = RAW / "harvard_glaucoma"

# Harvard HGDP — check project page
print("Harvard Glaucoma Dataset:")
print("  Paper: Luo et al., ICCV 2023")
print("  Check: https://ophai.hms.harvard.edu/datasets/harvard-glaucoma-detection-and-progression")
print("  Status:", "Downloaded" if HARVARD_DIR.exists() else "Not downloaded — manual registration may be required")

Harvard Glaucoma Dataset:
  Paper: Luo et al., ICCV 2023
  Check: https://ophai.hms.harvard.edu/datasets/harvard-glaucoma-detection-and-progression
  Status: Not downloaded — manual registration may be required


## 5. Next Steps

- **UWHVF** downloaded → go to `02_eda_uwhvf.ipynb` for exploration
- **GRAPE** downloaded → go to `03_eda_grape.ipynb` for exploration  
- Then run `04_vf_preprocessing.ipynb` to build the processed dataset